# The REG111_LICENSEE table

Welcome to table **REG111_LICENSEE** of PATSTAT Register. This table contains information about licensees and *rights in rem*.

In particular, we can find information about the country, the name and the address of the parties.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG111_LICENSEE
from sqlalchemy import func
import pandas as pd

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

## ID (Primary Key)

Technical identifier for an application, without business meaning. Its values will not change from one PATSTAT edition to the next.

In [2]:
i = db.query(
    REG111_LICENSEE.id
).limit(1000)

df = patstat.df(i)
df

,id
0,7865565
1,14845483
2,99971873
3,20204129
4,6711216
...,...
995,979964
996,95934692
997,99947109
998,4300872


## CHANGE_DATE

It is the date when the record was saved in the database.

In [3]:
change_date = db.query(
    REG111_LICENSEE.change_date,
    REG111_LICENSEE.id
).limit(100)

change_date_df = patstat.df(change_date)
change_date_df

,change_date,id
0,2011-12-23,7865565
1,2019-03-29,14845483
2,2004-10-15,99971873
3,2023-04-21,20204129
4,2012-11-23,6711216
...,...,...
95,2013-08-30,5256100
96,2005-09-09,98913107
97,2021-01-29,18206584
98,2007-11-09,2255070


## BULLETIN_YEAR

For actions that have been published in the European Patent Bulletin, it is the year of the publication in the European Patent Bulletin. The default value is 0, used for applications that are not published or for which the year is not known. The format is YYYY otherwise.

In [4]:
years = db.query(
    REG111_LICENSEE.bulletin_year,
    REG111_LICENSEE.id
).limit(1000)

years_df = patstat.df(years)
years_df

,bulletin_year,id
0,0,7865565
1,0,14845483
2,0,99971873
3,0,20204129
4,0,6711216
...,...,...
995,2003,979964
996,0,95934692
997,2003,99947109
998,2007,4300872


## BULLETIN_NR

This is the issue number of the European Patent Bulletin for actions that have been published in it. This attribute indicates the calendar week the European Patent Bulletin has been published. The default value 0 is used when the attribute `bulletin_year` is 0.

In [5]:
bulletin_nr = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.bulletin_nr,
    REG111_LICENSEE.bulletin_year
).limit(100)

bulletin_nr_df = patstat.df(bulletin_nr)
bulletin_nr_df

,id,bulletin_nr,bulletin_year
0,7865565,0,0
1,14845483,0,0
2,99971873,0,0
3,20204129,0,0
4,6711216,0,0
...,...,...,...
95,5256100,0,0
96,98913107,43,2005
97,18206584,0,0
98,2255070,0,0


## LICENSEE_SEQ_NR 

Serial number of license / sub license: the first two digits are the serial number of a main license and the optional other two digits represent the serial number of a sub-license. The maximum serial number for a main license and a sub-license is 10. Some applications may also have their licensee number set to 'deleted'.

In [6]:
licensee = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.licensee_seq_nr
).limit(100)

licensee_df = patstat.df(licensee)
licensee_df

,id,licensee_seq_nr
0,7865565,01
1,14845483,01
2,99971873,01
3,20204129,02
4,6711216,02
...,...,...
95,5256100,01
96,98913107,01 00
97,18206584,02
98,2255070,01


The total number of occurrences of the value 'deleted' is 102. This value for this attribute is found only in this table, but not in table `REG112_LICENSEE_STATES`.

In [7]:
deleted = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.licensee_seq_nr
).filter(
    REG111_LICENSEE.licensee_seq_nr == "deleted"
)

deleted_df = patstat.df(deleted)
deleted_df

,id,licensee_seq_nr
0,2714633,deleted
1,10815680,deleted
2,11187014,deleted
3,10754978,deleted
4,81110593,deleted
...,...,...
97,10807464,deleted
98,14828891,deleted
99,10011424,deleted
100,10782039,deleted


## TYPE_LICENSE

The type of the license: exclusive, not exclusive, or right *in rem*. The domain consists of 3 ASCII characters:
* 'NOL', i.e. no license (occurs if and only if `licensee_seq_nr` = ‘deleted’)
* 'EXC', i.e. exclusive licence
* 'NEX', i.e. non-exclusive licence
* 'RIR', i.e. right in rem

We can check the one-to-one relation between the values 'NOL' and 'deleted' from attributes `type_license` and `licensee_seq_nr` respectively.

In [8]:
nol = db.query(
    REG111_LICENSEE.type_license,
    func.count(REG111_LICENSEE.id).label('number of applications license')
).group_by(
    REG111_LICENSEE.type_license
).filter(
    REG111_LICENSEE.licensee_seq_nr == 'deleted'
)

nol_df = patstat.df(nol)
nol_df

,type_license,number of applications license
0,NOL,102


We can see that the number of applications without license (i.e. marked as 'NOL') are equal to the number of `licensee_seq_nr` equal to 'deleted'. Therefore, there are 102 applications without license.

## DESIGNATION

Type of designation. Default value apart, i.e. empty string, this attribute can assume two values: 'all' and 'as-indicated'.

In [9]:
desig = db.query(
    REG111_LICENSEE.designation,
    func.count(REG111_LICENSEE.id).label('number_of_applications')
).group_by(
    REG111_LICENSEE.designation
).order_by(
    func.count(REG111_LICENSEE.id).desc()
)

desig_df = patstat.df(desig)
desig_df

,designation,number_of_applications
0,as-indicated,10729
1,all,4545


## VALID_DATE

This attribute indicates the date when the license became valid.

We can check, for example, how many and which licenses became valid after 2015.

In [10]:
valid_date = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.valid_date
).filter(
    REG111_LICENSEE.valid_date > '2015-12-31',
    REG111_LICENSEE.valid_date != '9999-12-31'
)

valid_date_df = patstat.df(valid_date)
valid_date_df

,id,valid_date
0,14845483,2018-10-10
1,20204129,2023-02-24
2,17170282,2021-01-21
3,17831691,2020-04-30
4,12769755,2019-12-20
...,...,...
3488,20174099,2021-02-10
3489,23191512,2024-05-15
3490,19730153,2024-05-15
3491,14723648,2020-05-11


## CUSTOMER_ID

The identifier of a customer, which can be a legal or natural person. If a customer has several roles (applicant, inventor, representative, licensee, opponent) he usually has several `customer_id`s. 

Not every person has a `customer_id` assigned. Persons which are only inventors (and not e. g. also applicants) do not have a `customer_id`. 

In [11]:
customer = db.query(
    REG111_LICENSEE.customer_id
).filter(
    REG111_LICENSEE.customer_id != ""
)

customer_df = patstat.df(customer)
customer_df

,customer_id
0,0101221887
1,0101078366
2,0101103733
3,0100725748
4,0100726154
...,...
5587,0101279544
5588,0101933269
5589,0100725633
5590,0100726364


Let's count how many persons are not inventors.

In [12]:
non_inventors = db.query(
    func.count(REG111_LICENSEE.id).label('non inventors')
).filter(
    REG111_LICENSEE.customer_id == ""
)

non_inventors_df = patstat.df(non_inventors)
print(f"There are {non_inventors_df['non inventors'].item()} applications without customer ID.")

There are 9682 applications without customer ID.


## NAME

Name of a person (natural person or legal person).

In showing the names, we have to filter out the empty spaces (default value).

In [13]:
name = db.query(
    REG111_LICENSEE.name
).filter(
    REG111_LICENSEE.name != ""
).limit(100)

name_df = patstat.df(name)
name_df

,name
0,Sigma Offshore Limited
1,Mitutoyo Corporation
2,Delft Patents B.V.
3,LEGRAND SNC
4,"ROCKWOOL ISOLATION S.A., société anonyme"
...,...
95,Société dite Industrie Distribution Service (IDS)
96,Zumtobel Lighting GmbH & Co. KG
97,"ROCKWOOL ISOLATION S.A., société anonyme"
98,ATELIERS L.R. ETANCO


We can see that there quite many empty names.

In [14]:
empty = db.query(
    func.count(REG111_LICENSEE.name).label('empty names')
).filter(
    REG111_LICENSEE.name == ""
)

empty_df = patstat.df(empty)
print(f"There are {empty_df['empty names'].item()} empty names.")

There are 9682 empty names.


## ADDRESS_1

First address line of a person. Default value: empty. In PATSTAT Online these attributes are not populated.

In [15]:
addr1 = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.name,
    REG111_LICENSEE.address_1
).limit(100)

addr1_df = patstat.df(addr1)
addr1_df

,id,name,address_1
0,7865565,,
1,14845483,,
2,99971873,,
3,20204129,,
4,6711216,,
...,...,...,...
95,5256100,,
96,98913107,Candescent Technologies Corporation,
97,18206584,,
98,2255070,,


## ADDRESS_2

Second address line of a person. Default value: empty. In PATSTAT Online these attributes are not populated.

In [16]:
addr2 = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.name,
    REG111_LICENSEE.address_2
).limit(100)

addr2_df = patstat.df(addr2)
addr2_df

,id,name,address_2
0,7865565,,
1,14845483,,
2,99971873,,
3,20204129,,
4,6711216,,
...,...,...,...
95,5256100,,
96,98913107,Candescent Technologies Corporation,
97,18206584,,
98,2255070,,


## ADDRESS_3

Third address line of a person. Default value: empty. In PATSTAT Online these attributes are not populated.

In [17]:
addr3 = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.name,
    REG111_LICENSEE.address_3
).filter(
    REG111_LICENSEE.address_3 != ""
)

addr3_df = patstat.df(addr3)
addr3_df

""


## ADDRESS_4

Fourth address line of a person. Default value: empty. In PATSTAT Online these attributes are not populated.

In [18]:
addr4 = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.name,
    REG111_LICENSEE.address_4
).filter(
    REG111_LICENSEE.address_4 != ""
)

addr4_df = patstat.df(addr4)
addr4_df

""


## ADDRESS_5

Fifth address line of a person. Default value: empty. In PATSTAT Online these attributes are not populated.

In [19]:
addr5 = db.query(
    REG111_LICENSEE.id,
    REG111_LICENSEE.name,
    REG111_LICENSEE.address_5
).filter(
    REG111_LICENSEE.address_5 != ""
)

addr5_df = patstat.df(addr5)
addr5_df

""


## COUNTRY

Two-letter country/territory code for patent parties (applicant/inventor/agent), designated states of applicant, or country of licensees. Default value: empty. The domain consists of up to 2 alphabetic characters, according to WIPO ST.3, plus minor additions.

We rank the most frequent countries in the database, excluding the default value, which is a double space.

In [20]:
country = db.query(
    REG111_LICENSEE.country,
    func.count(REG111_LICENSEE.id).label('number_of_applications')
).filter(
    REG111_LICENSEE.country != "  "
).group_by(
    REG111_LICENSEE.country
).order_by(
    func.count(REG111_LICENSEE.id).desc()
).limit(15)

country_df = patstat.df(country)
country_df

,country,number_of_applications
0,FR,1783
1,GB,1367
2,US,803
3,DE,250
4,NL,198
5,IT,178
6,IE,137
7,CH,129
8,BE,101
9,JP,87
